# 01 · Country features — master input table

**Data stream:** country (location) features — geography, climate, socio-economy.
**Output feeds:** the country-feature branch of the invasion model (`country_features_model.csv`),
and notebooks 02 (ISO3 codes and population for the trade allocation) and 04 (location list).

---

## What this notebook does

Every location in SInAS (a country, or a sub-national unit such as an island
group or a US state) gets **one row** with **one column per variable**. Earlier
versions mixed our own variables with columns from Manuela's file, so the same
concept (e.g. latitude, area, temperature seasonality) existed two or three
times, computed in different ways. This notebook fixes that:

| Concept | Single definition used here |
|---|---|
| Coordinates | Centroid of the SInAS polygon (equal-area projection) |
| Area / perimeter | Computed from the SInAS polygon (EPSG:6933) |
| All climate | WorldClim 2.1, 1970–2000 means, 2.5 arc-min, polygon-weighted mean. Locations outside the WorldClim land mask (currently only Saint Helena) are gap-filled from CHELSA V2.1 and flagged |
| Frost days | Continuous estimate from monthly Tmin (see Step 4) |
| Aridity | Annual precipitation / Hargreaves PET |
| Climate *distribution* within a country | Area-weighted p05 / p50 / p95 of 5 key variables, and the area fraction in each of the 30 Köppen–Geiger classes (computed from WorldClim monthly T and P) |
| Socio-economy | Manuela's location table (GDP, population density, colonial years, native richness, island/coastal) |

## Pipeline at a glance

| Step | What happens | Main output |
|---|---|---|
| 0 | Configure paths, download/check raw data | — |
| 1 | Read SInAS: authoritative location names + diagnostic counts | `loc_names`, `diag` |
| 2 | Read Manuela's table, collapse to one row per location | `loc_tbl` |
| 3 | Read polygons, compute geometry metrics, merge 1+2 | `gdf` |
| 4a | WorldClim zonal statistics (means, quantiles) | `results` |
| 4b | CHELSA gap-fill for locations WorldClim does not cover | filled `results` |
| 4c | Climate columns + derived indices | climate columns in `gdf` |
| 4d | Within-country climate distribution: quantiles + Köppen–Geiger fractions | `*_p05/p50/p95`, `kg_*` columns |
| 5 | Quality assurance (name coverage, cross-checks, missing data) | `qa` dict |
| 6 | Write master table, model-ready table, schema, QA report | files in `data/` |

## Outputs (written to `data/`)

* `complete_country_features_master.csv` / `.gpkg` — every variable once, plus
  `qa_*` (extraction flags) and `diag_*` (label-derived counts) columns.
* `country_features_model.csv` — **model-ready**: no `diag_*`/`qa_*` columns,
  skewed variables log1p-transformed, missing values median-imputed with a
  `<col>_missing` indicator.
* `country_koppen_fractions.csv` — area fraction of each location in each Köppen–Geiger class (raw, unimputed), for building species climate profiles in the model notebook.
* `country_features_schema.json` — feature groups, transforms, imputation log.
* `country_features_qa.json` — QA report (see Step 5).

## Design rules (read before editing)

1. **Climate comes from one source** (WorldClim 2.1, 1970–2000, 2.5').
   The only exception: locations WorldClim does not cover at all are
   gap-filled from CHELSA V2.1 (1981–2010) and flagged
   `qa_wc_extract_method = "chelsa_gapfill"`. Manuela's `temp_season` / `precip_season` are used *only* as a QA
   cross-check in Step 5 and are then dropped.
2. **Each concept exists once** — no alias columns.
3. **`diag_*` columns are derived from SInAS labels** (how many alien species
   a location has). They are kept for inspection but are **never model inputs**
   — that would leak the target into the features.
4. **`location` is the SInAS raw name** — the key used by the trade tensor,
   GBIF and species files. Do not rename it.
5. Scaling (min-max / standardisation) is **not** done here; do it inside the
   model pipeline, fitted on training locations only.
6. **Pixels are weighted by area**: covered fraction × cos(latitude), so a
   high-latitude pixel (which covers less ground) counts for less. This
   matters for large countries spanning many latitudes (Canada, Russia).

## How to run

1. Install dependencies (once):
   `pip install geopandas rasterio exactextract scipy pandas numpy tqdm`
2. Put the three raw input files in `../data/` (see Step 0 for download links).
3. *Run All*. The first run downloads ~1.2 GB of WorldClim rasters; zonal
   extraction takes roughly 5–15 min depending on the machine.

A full description of every variable and its provenance is in
`country_features_metadata.md`.

## Step 0 · Data sources and configuration

### Raw inputs

| File | Source | How to obtain |
|---|---|---|
| `SInAS_3.1.1.csv` | SInAS v3.1.1 — Gómez-Suárez, Laeseke & Seebens (2025), *Scientific Data* 12: 1914, [doi:10.1038/s41597-025-06379-6](https://doi.org/10.1038/s41597-025-06379-6) | **Manual download** from Zenodo: <https://zenodo.org/records/17727120> (record "Version 3.1.1 + geopackage") |
| `SInAS_locations_3.1.gpkg` | SInAS location polygons (corrected geopackage) | **Manual download**, same Zenodo record as above |
| `variables_SInAS_manuela.csv` | Location-level covariates compiled by Manuela Gómez-Suárez (GDP per capita, population density, colonial years, native richness per taxon group, island/coastal flags) | **Internal file** — obtained directly from Manuela; not public. Original sources are listed in `country_features_metadata.md` |
| CHELSA V2.1 climatologies 1981–2010 (gap-fill only) | Karger et al. (2017) *Scientific Data* 4: 170122, [doi:10.1038/sdata.2017.122](https://doi.org/10.1038/sdata.2017.122); data: [doi:10.16904/envidat.228](https://doi.org/10.16904/envidat.228) | **Automatic, nothing to download**: only the pixels of gap-filled polygons are read directly from the CHELSA cloud server (<https://www.chelsa-climate.org/datasets/chelsa_climatologies>) |
| `climate_rasters/wc2.1_2.5m_*.tif` | WorldClim 2.1 historical climate, 1970–2000 — Fick & Hijmans (2017), *Int. J. Climatol.* 37: 4302–4315 | **Automatic** download below (`DOWNLOAD_RASTERS = True`). Manual alternative: <https://www.worldclim.org/data/worldclim21.html> → resolution *2.5 minutes* → `bio`, `tmin`, `tmax`, `prec`, `elev`; unzip into `data/climate_rasters/` |

Optional: `sinas_matched_species.csv` (output of notebook 03) — used only in
Step 5 to check that every location the model uses has a feature row.

In [1]:
import csv
import json
import re
import urllib.request
import warnings
import zipfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from exactextract import exact_extract
from rasterio.features import rasterize
from rasterio.transform import Affine
from rasterio.windows import Window, from_bounds
from scipy.special import ndtr
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 50)

### Configuration

All paths and tunable parameters live here. The parameters are explained
inline; you should normally only need to change `DATA`.

In [2]:
# ---------------------------------------------------------------- paths
DATA = Path.cwd().parent / "data"
SINAS_RAW_CSV = DATA / "SInAS_3.1.1.csv"
MANUELA_CSV = DATA / "variables_SInAS_manuela.csv"
GPKG_FILE = DATA / "SInAS_locations_3.1.gpkg"
RASTER_DIR = DATA / "climate_rasters"

# Optional: species file from notebook 03, used for name-coverage QA only.
MODEL_SPECIES_CANDIDATES = [
    DATA / "sinas_matched_species.csv",
    DATA.parent / "sinas_matched_species.csv",
]

# ---------------------------------------------------------------- outputs
CSV_OUT = DATA / "complete_country_features_master.csv"
GPKG_OUT = DATA / "complete_country_features_master.gpkg"
MODEL_FEATURES_OUT = DATA / "country_features_model.csv"
SCHEMA_OUT = DATA / "country_features_schema.json"
QA_OUT = DATA / "country_features_qa.json"
KOPPEN_OUT = DATA / "country_koppen_fractions.csv"

# ---------------------------------------------------------------- climate
DOWNLOAD_RASTERS = True  # fetch WorldClim zips if they are missing
WC_RES = "2.5m"  # WorldClim resolution used for ALL climate variables
WC_BASE_URL = (
    "https://geodata.ucdavis.edu/climate/worldclim/2_1/base/"
    "wc2.1_" + WC_RES + "_{}.zip"
)
WC_LAYERS = ("bio", "tmin", "tmax", "prec", "elev")

# Within-month standard deviation of daily Tmin (deg C), used to turn a
# monthly mean Tmin into an expected number of frost days.
FROST_SIGMA_C = 3.5
# Lower bound on PET (mm/yr) so the aridity ratio does not explode in
# polar / frozen regions where Hargreaves PET approaches zero.
PET_FLOOR_MM = 10.0

# Small islands / atolls may contain no valid land pixel. We then take the
# nearest valid pixel, searching within these radii.
PX_KM = 4.6  # approximate 2.5' pixel size at the equator (km)
FALLBACK_RADII_KM = (15, 45, 115)
FALLBACK_RADII_PX = tuple(
    max(1, int(np.ceil(km / PX_KM))) for km in FALLBACK_RADII_KM
)

# CHELSA V2.1 gap-fill for polygons outside the WorldClim land mask.
# Pixels are read straight from the CHELSA cloud server (no download).
CHELSA_GAPFILL = True
CHELSA_BASE_URLS = [  # tried in order; first one that answers is used
    "https://os.zhdk.cloud.switch.ch/chelsav2/GLOBAL/climatologies",
    "https://os.zhdk.cloud.switch.ch/envicloud/chelsa/chelsa_V2/GLOBAL/"
    "climatologies",
]
CHELSA_PERIOD = "1981-2010"
CHELSA_N_CONTROLS = 3  # nearby WorldClim-covered islands for the QA check
CHELSA_SUPERSAMPLE = 10  # sub-pixel grid for polygon coverage fractions

# Within-country climate distribution: area-weighted quantiles of these
# BIOCLIM variables (1 annual mean T, 5 max T warmest month, 6 min T
# coldest month, 12 annual precipitation, 14 driest-month precipitation).
QUANTILE_VARS = (1, 5, 6, 12, 14)
QUANTILES = (0.05, 0.50, 0.95)

# Polygons are simplified (for climate extraction only) to this tolerance in
# degrees. 0.005 deg ~ 500 m ~ a tenth of a 2.5' pixel: invisible to the
# rasters, but it removes millions of coastline vertices. 0 = no simplify.
SIMPLIFY_DEG = 0.005

for f in (SINAS_RAW_CSV, MANUELA_CSV, GPKG_FILE):
    status = "OK " if f.exists() else "MISSING"
    print(f"[{status}] {f}")

[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\SInAS_3.1.1.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\variables_SInAS_manuela.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\SInAS_locations_3.1.gpkg


### Helper functions

Grouped into (a) file reading, (b) raster handling and (c) derived climate
indices. Each function has a docstring explaining its inputs and outputs.
Nothing is computed yet — the steps below call these functions.

In [3]:
# ======================================================================
# (a) File reading and string helpers
# ======================================================================
def first_existing(*paths):
    """Return the first path that exists, or None if none do."""
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def read_table_auto(path, **kwargs):
    """Read a delimited text file whose delimiter/encoding is unknown.

    The delimiter is detected with :class:`csv.Sniffer` on the first 64 kB
    of the file, choosing among comma, semicolon, tab, pipe and **space**.
    Space matters here: SInAS is written by R's ``write.table`` with
    space-separated, double-quoted fields. If sniffing fails, pandas'
    own detector (``sep=None``) is used. The fast C parser is tried first;
    if some rows have the wrong number of fields, the python parser is used
    instead and ONLY those rows are skipped, with a warning.

    Parameters
    ----------
    path : str or Path
        File to read.
    **kwargs
        Passed on to :func:`pandas.read_csv`.

    Returns
    -------
    pandas.DataFrame
    """
    path = Path(path)
    last_err = None
    for enc in ("utf-8-sig", "latin-1"):
        try:
            with open(path, "r", encoding=enc, newline="") as fh:
                sample = fh.read(65536)
            sample = sample[:sample.rfind("\n") + 1] or sample
            try:
                sep = csv.Sniffer().sniff(sample,
                                          delimiters=",;\t| ").delimiter
            except csv.Error:
                sep = None  # let pandas guess
            print(f"   {path.name}: encoding={enc}, sep={sep!r}")

            common = dict(encoding=enc, quotechar='"', **kwargs)
            if sep is None:
                return pd.read_csv(path, sep=None, engine="python",
                                   **common)
            try:
                return pd.read_csv(path, sep=sep, low_memory=False,
                                   **common)
            except pd.errors.ParserError as err:
                print(f"   C parser failed ({err}); retrying with the "
                      "python engine and skipping malformed rows")
                bad_rows = []

                def _skip(fields):
                    bad_rows.append(fields)
                    return None

                df = pd.read_csv(path, sep=sep, engine="python",
                                 on_bad_lines=_skip, **common)
                if bad_rows:
                    warnings.warn(
                        f"{path.name}: skipped {len(bad_rows)} malformed "
                        f"rows (first: "
                        f"{[str(r)[:150] for r in bad_rows[:2]]})"
                    )
                return df
        except UnicodeDecodeError as err:
            last_err = err
    raise ValueError(f"Could not decode {path}: {last_err}")


def norm_name(s):
    """Case- and whitespace-insensitive key for fuzzy name comparison."""
    return re.sub(r"\s+", " ", str(s)).strip().casefold()


def slug(s):
    """Turn a free-text label into a safe lower_snake_case column suffix."""
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(s)).strip("_").lower()

In [4]:
# ======================================================================
# (b) WorldClim raster handling
# ======================================================================
# Human-readable column names for the 19 bioclimatic variables.
BIOCLIM_NAMES = {
    1: "wc_bio01_annual_mean_temp_C",
    2: "wc_bio02_mean_diurnal_range_C",
    3: "wc_bio03_isothermality",
    4: "wc_bio04_temp_seasonality_sd_C",
    5: "wc_bio05_max_temp_warmest_month_C",
    6: "wc_bio06_min_temp_coldest_month_C",
    7: "wc_bio07_temp_annual_range_C",
    8: "wc_bio08_mean_temp_wettest_quarter_C",
    9: "wc_bio09_mean_temp_driest_quarter_C",
    10: "wc_bio10_mean_temp_warmest_quarter_C",
    11: "wc_bio11_mean_temp_coldest_quarter_C",
    12: "wc_bio12_annual_precipitation_mm",
    13: "wc_bio13_precip_wettest_month_mm",
    14: "wc_bio14_precip_driest_month_mm",
    15: "wc_bio15_precip_seasonality_cv",
    16: "wc_bio16_precip_wettest_quarter_mm",
    17: "wc_bio17_precip_driest_quarter_mm",
    18: "wc_bio18_precip_warmest_quarter_mm",
    19: "wc_bio19_precip_coldest_quarter_mm",
}


def ensure_rasters():
    """Download and unzip any missing WorldClim layer into RASTER_DIR.

    A layer counts as present if at least one matching ``.tif`` exists
    anywhere under RASTER_DIR (subfolders included); an existing zip is
    unzipped without downloading again. Set ``DOWNLOAD_RASTERS = False`` to
    skip this check entirely once the rasters are in place. Download
    failures are reported (not raised) so you can fetch the zip manually;
    :func:`index_rasters` will then fail loudly if files are still missing.
    """
    RASTER_DIR.mkdir(parents=True, exist_ok=True)
    if not DOWNLOAD_RASTERS:
        return
    tif_names = [p.name for p in RASTER_DIR.rglob("*.tif")]
    for name in WC_LAYERS:
        pattern = rf"{re.escape(WC_RES)}_{name}.*\.tif$"
        if any(re.search(pattern, n) for n in tif_names):
            continue
        dest = RASTER_DIR / f"wc2.1_{WC_RES}_{name}.zip"
        try:
            if not dest.exists():
                print(f"   downloading {dest.name} ...")
                urllib.request.urlretrieve(WC_BASE_URL.format(name), dest)
            print(f"   unzipping {dest.name} (can take a few minutes) ...",
                  flush=True)
            with zipfile.ZipFile(dest) as zf:
                zf.extractall(RASTER_DIR)
        except Exception as err:  # network error or host moved
            print(f"   could not fetch {name}: {err}\n"
                  f"   -> download it manually into {RASTER_DIR}")


def index_rasters():
    """Locate every required WorldClim GeoTIFF.

    Returns
    -------
    dict
        ``{'bio': {1..19: path}, 'tmin': {1..12: path},
        'tmax': {1..12: path}, 'prec': {1..12: path}, 'elev': path}``

    Raises
    ------
    RuntimeError
        If any layer is missing or duplicated.
    """
    pool = [p for p in RASTER_DIR.rglob("*.tif") if f"_{WC_RES}_" in p.name]
    if not pool:
        raise RuntimeError(f"No *_{WC_RES}_* rasters in {RASTER_DIR}")
    patterns = {
        "bio": r"bio_?(\d+)\.tif$",
        "tmin": r"tmin_?(\d+)\.tif$",
        "tmax": r"tmax_?(\d+)\.tif$",
        "prec": r"prec_?(\d+)\.tif$",
    }
    found = {k: {} for k in patterns}
    for key, pat in patterns.items():
        for p in pool:
            m = re.search(pat, p.name, re.IGNORECASE)
            if m:
                n = int(m.group(1))
                if n in found[key]:
                    raise RuntimeError(f"Duplicate {key}{n}: "
                                       f"{found[key][n]} vs {p}")
                found[key][n] = p
    elev = [p for p in pool if re.search(r"elev\.tif$", p.name, re.I)]
    found["elev"] = elev[0] if elev else None

    problems = []
    if sorted(found["bio"]) != list(range(1, 20)):
        problems.append(f"bio: found {sorted(found['bio'])}")
    for key in ("tmin", "tmax", "prec"):
        if sorted(found[key]) != list(range(1, 13)):
            problems.append(f"{key}: found {sorted(found[key])}")
    if found["elev"] is None:
        problems.append("elev: not found")
    if problems:
        raise RuntimeError(f"WorldClim rasters incomplete in {RASTER_DIR}: "
                           + "; ".join(problems))
    return found


def _clean(values):
    """Return a float copy with WorldClim nodata (~-3.4e38) set to NaN."""
    arr = np.array(values, dtype="float64")
    arr[~np.isfinite(arr)] = np.nan
    arr[arr < -1e30] = np.nan
    return arr


def nearest_valid(src, x, y, radii=FALLBACK_RADII_PX):
    """Value of the valid pixel nearest to (x, y) in an open raster.

    Searches square windows of increasing radius (in pixels) and stops at
    the first window containing a valid value.

    Returns
    -------
    tuple of (float, float)
        ``(value, distance_in_pixels)``; ``(nan, nan)`` if nothing found.
    """
    r0, c0 = src.index(x, y)
    for rad in radii:
        r_lo, c_lo = max(r0 - rad, 0), max(c0 - rad, 0)
        r_hi = min(r0 + rad + 1, src.height)
        c_hi = min(c0 + rad + 1, src.width)
        if r_lo >= r_hi or c_lo >= c_hi:
            continue
        window = Window(c_lo, r_lo, c_hi - c_lo, r_hi - r_lo)
        arr = _clean(src.read(1, window=window))
        if src.nodata is not None and np.isfinite(src.nodata):
            arr[arr == src.nodata] = np.nan
        valid = np.argwhere(np.isfinite(arr))
        if len(valid):
            dist = np.hypot(valid[:, 0] + r_lo - r0,
                            valid[:, 1] + c_lo - c0)
            k = int(dist.argmin())
            return float(arr[valid[k, 0], valid[k, 1]]), float(dist[k])
    return np.nan, np.nan


def build_coverage_index(zones, ref_raster):
    """Record which raster cells each polygon covers, and by how much.

    This is the expensive geometric step, so it is done ONCE with
    ``exactextract`` and then reused for every WorldClim layer (they all
    share the same grid).

    Parameters
    ----------
    zones : geopandas.GeoDataFrame
        Polygons (clean RangeIndex, EPSG:4326).
    ref_raster : Path
        Any raster on the target grid (e.g. BIO1).

    Returns
    -------
    dict
        ``owner`` (polygon index per covered cell), ``cell_id`` (flat
        raster index), ``coverage`` (covered fraction 0-1), ``area_w``
        (coverage x cos(latitude) = relative area inside the polygon),
        ``n`` (number of polygons), ``shape`` (raster shape) and ``ref``.
    """
    res = exact_extract(str(ref_raster), zones, ["cell_id", "coverage"],
                        output="pandas", progress=True)
    id_col = next(c for c in res.columns if c.endswith("cell_id"))
    cov_col = next(c for c in res.columns if c.endswith("coverage"))
    cell_ids = [np.asarray(a, dtype=np.int64) for a in res[id_col]]
    with rasterio.open(ref_raster) as src:
        shape, transform = src.shape, src.transform
    cell_id = np.concatenate(cell_ids)
    coverage = np.concatenate([np.asarray(a, dtype=float)
                               for a in res[cov_col]])
    # pixel area shrinks with cos(latitude) on a lat/lon grid
    row_lat = transform.f + (cell_id // shape[1] + 0.5) * transform.e
    return {
        "owner": np.repeat(np.arange(len(zones)),
                           [len(a) for a in cell_ids]),
        "cell_id": cell_id,
        "coverage": coverage,
        "area_w": coverage * np.cos(np.radians(row_lat)),
        "n": len(zones),
        "shape": shape,
        "ref": Path(ref_raster),
    }


def read_grid(path, index):
    """Read a whole raster as a flat float64 array (nodata -> NaN).

    Checks that the raster is on the same grid as the coverage index.
    """
    with rasterio.open(path) as src:
        if src.shape != index["shape"]:
            raise ValueError(f"{Path(path).name} is not on the same grid "
                             f"as {index['ref'].name}")
        flat = _clean(src.read(1).ravel())
        if src.nodata is not None and np.isfinite(src.nodata):
            flat[flat == src.nodata] = np.nan
    return flat


def zonal_from_index(path, index, stat="mean", pts=None, quantiles=()):
    """Zonal statistic per polygon, using a precomputed coverage index.

    Reads the whole raster once (~300 MB as float64) and aggregates the
    covered cells per polygon with numpy, so each layer takes seconds.
    ``mean`` is area-weighted (covered fraction x cos(latitude));
    ``min``/``max`` use every cell touched by the polygon. Nodata cells
    are ignored. Polygons without any valid cell get the value of the
    nearest valid pixel to their point in ``pts`` (atolls, tiny islands).

    Parameters
    ----------
    path : Path
        GeoTIFF on the same grid as ``index['ref']``.
    index : dict
        Output of :func:`build_coverage_index`.
    stat : {"mean", "min", "max"}
    pts : geopandas.GeoSeries, optional
        One point inside each polygon, for the nearest-pixel fallback.
    quantiles : sequence of float, optional
        Also return these area-weighted quantiles per polygon.

    Returns
    -------
    tuple
        ``(values, fallback_used, fallback_distance_px, quantile_values)``;
        ``quantile_values`` has shape (len(quantiles), n) or is None.
    """
    n, owner, w_area = index["n"], index["owner"], index["area_w"]
    v = read_grid(path, index)[index["cell_id"]]
    ok = np.isfinite(v)

    if stat == "mean":
        num = np.bincount(owner[ok], weights=v[ok] * w_area[ok],
                          minlength=n)
        den = np.bincount(owner[ok], weights=w_area[ok], minlength=n)
        vals = np.divide(num, den, out=np.full(n, np.nan), where=den > 0)
    elif stat in ("min", "max"):
        func, start = ((np.minimum, np.inf) if stat == "min"
                       else (np.maximum, -np.inf))
        vals = np.full(n, start)
        func.at(vals, owner[ok], v[ok])
        vals[np.isinf(vals)] = np.nan
    else:
        raise ValueError(f"Unsupported stat: {stat}")
    qvals = (weighted_group_quantiles(v, w_area, owner, n, quantiles)
             if len(quantiles) else None)

    used_fb = np.zeros(n, dtype=bool)
    dist = np.full(n, np.nan)
    missing = np.where(np.isnan(vals))[0]
    if len(missing) and pts is not None:
        with rasterio.open(path) as src:
            for i in missing:
                val, d = nearest_valid(src, pts.iloc[i].x, pts.iloc[i].y)
                if np.isfinite(val):
                    vals[i], used_fb[i], dist[i] = val, True, d
    if qvals is not None:  # single fallback pixel -> all quantiles equal
        qvals[:, used_fb] = vals[used_fb]
    return vals, used_fb, dist, qvals

In [5]:
# ======================================================================
# (d) CHELSA V2.1 gap-fill (remote, windowed reads)
# ======================================================================
# GDAL settings for fast remote reads of cloud-optimised GeoTIFFs.
CHELSA_GDAL_ENV = {
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "3",
    "GDAL_HTTP_RETRY_DELAY": "2",
}
_CHELSA_BASE = None  # cached working base URL
TEMPERATURE_LAYERS = {("bio", i) for i in (1, 5, 6, 8, 9, 10, 11)}


def chelsa_url(layer, idx):
    """Remote GDAL path of a CHELSA V2.1 1981-2010 climatology.

    ``layer`` is ``"bio"`` (idx 1-19), ``"tmin"``, ``"tmax"`` or
    ``"prec"`` (idx = month 1-12). The first base URL in CHELSA_BASE_URLS
    that can be opened is cached and reused.
    """
    global _CHELSA_BASE
    if layer == "bio":
        rel = (f"{CHELSA_PERIOD}/bio/"
               f"CHELSA_bio{idx}_{CHELSA_PERIOD}_V.2.1.tif")
    else:
        var = {"tmin": "tasmin", "tmax": "tasmax", "prec": "pr"}[layer]
        rel = (f"{CHELSA_PERIOD}/{var}/"
               f"CHELSA_{var}_{idx:02d}_{CHELSA_PERIOD}_V.2.1.tif")
    bases = [_CHELSA_BASE] if _CHELSA_BASE else CHELSA_BASE_URLS
    errors = []
    for base in bases:
        path = f"/vsicurl/{base}/{rel}"
        try:
            with rasterio.Env(**CHELSA_GDAL_ENV), rasterio.open(path):
                _CHELSA_BASE = base
                return path
        except rasterio.errors.RasterioIOError as err:
            errors.append(f"{base}: {err}")
    raise RuntimeError(
        f"CHELSA file not reachable: {rel}\n" + "\n".join(errors)
        + "\nCheck https://www.chelsa-climate.org/datasets/"
        "chelsa_climatologies for the current server and update "
        "CHELSA_BASE_URLS."
    )


PLAUSIBLE_RANGE = {"temperature": (-90.0, 60.0),
                   "precip_month": (0.0, 5000.0)}


def chelsa_polygon_pixels(path, geoms, kind=None):
    """Pixel values and area weights of a remote CHELSA layer per polygon.

    Only the pixels inside each polygon's bounding box are downloaded.
    Each pixel's weight is the fraction of the pixel inside the polygon
    (estimated on a CHELSA_SUPERSAMPLE x CHELSA_SUPERSAMPLE sub-grid) times
    cos(latitude), matching the WorldClim weighting. The file's own
    scale/offset metadata converts stored integers to physical units
    (e.g. 0.1 * value - 273.15 for temperatures in deg C). All CHELSA
    layers share one grid, so for a given polygon every layer returns its
    pixels in the same order.

    Parameters
    ----------
    path : str
        GDAL path from :func:`chelsa_url`.
    geoms : sequence of shapely geometries (EPSG:4326)
    kind : {"temperature", "precip_month", None}
        Raise if a value falls outside PLAUSIBLE_RANGE[kind]; this catches
        scale/offset (unit) errors.

    Returns
    -------
    list of (numpy.ndarray, numpy.ndarray)
        ``(values, weights)`` per geometry; NaN for invalid pixels.
    """
    out = []
    with rasterio.Env(**CHELSA_GDAL_ENV), rasterio.open(path) as src:
        scale = src.scales[0] if src.scales[0] else 1.0
        offset = src.offsets[0] if src.offsets[0] else 0.0
        for geom in geoms:
            w = from_bounds(*geom.bounds, transform=src.transform)
            c0 = max(int(np.floor(w.col_off)) - 1, 0)
            r0 = max(int(np.floor(w.row_off)) - 1, 0)
            c1 = min(int(np.ceil(w.col_off + w.width)) + 1, src.width)
            r1 = min(int(np.ceil(w.row_off + w.height)) + 1, src.height)
            if c0 >= c1 or r0 >= r1:
                out.append((np.array([]), np.array([])))
                continue
            win = Window(c0, r0, c1 - c0, r1 - r0)
            raw = src.read(1, window=win).astype("float64")
            bad = ~np.isfinite(raw)
            if src.nodata is not None:
                bad |= raw == src.nodata
            if src.dtypes[0] == "uint16":
                bad |= raw == 65535  # uint16 fill value
            h, wd = raw.shape
            f = CHELSA_SUPERSAMPLE if h * wd < 4_000_000 else 1
            win_tf = src.window_transform(win)
            fine = rasterize(
                [(geom, 1)], out_shape=(h * f, wd * f),
                transform=win_tf * Affine.scale(1 / f),
                fill=0, dtype="uint8",
            )
            frac = fine.reshape(h, f, wd, f).mean(axis=(1, 3))
            lat = win_tf.f + (np.arange(h) + 0.5) * win_tf.e
            weight = frac * np.cos(np.radians(lat))[:, None]
            vals = np.where(bad, np.nan, raw * scale + offset)
            keep = weight.ravel() > 0
            out.append((vals.ravel()[keep], weight.ravel()[keep]))
    if kind is not None:
        lo, hi = PLAUSIBLE_RANGE[kind]
        allv = np.concatenate([v for v, _ in out]) if out else np.array([])
        allv = allv[np.isfinite(allv)]
        if allv.size and ((allv < lo).any() or (allv > hi).any()):
            raise ValueError(f"Implausible CHELSA {kind} in {path}: "
                             f"range {allv.min():.1f}..{allv.max():.1f} "
                             f"(scale={scale}, offset={offset})")
    return out


def weighted_mean(values, weights):
    """Weighted mean ignoring NaN values (NaN if nothing is valid)."""
    ok = np.isfinite(values) & (weights > 0)
    if not ok.any():
        return np.nan
    return float((values[ok] * weights[ok]).sum() / weights[ok].sum())


# ======================================================================
# (e) Climate-distribution features: weighted quantiles and Koppen-Geiger
# ======================================================================
# The 30 Koppen-Geiger classes in the order of Beck et al. (2018).
KOPPEN_CLASSES = [
    "Af", "Am", "Aw", "BWh", "BWk", "BSh", "BSk",
    "Csa", "Csb", "Csc", "Cwa", "Cwb", "Cwc", "Cfa", "Cfb", "Cfc",
    "Dsa", "Dsb", "Dsc", "Dsd", "Dwa", "Dwb", "Dwc", "Dwd",
    "Dfa", "Dfb", "Dfc", "Dfd", "ET", "EF",
]
_KG_CODE = {name: k for k, name in enumerate(KOPPEN_CLASSES)}


def weighted_group_quantiles(values, weights, groups, n_groups, qs):
    """Weighted quantiles of ``values`` within each group.

    Used for the within-country climate distribution: ``groups`` is the
    polygon index of each pixel and ``weights`` its covered area, so the
    p05 of a country is the value below which 5 % of its *area* lies.

    Parameters
    ----------
    values, weights : numpy.ndarray, shape (m,)
        Pixel values and non-negative weights. NaN values are ignored.
    groups : numpy.ndarray of int, shape (m,)
        Group (polygon) index of each pixel, 0 .. n_groups - 1.
    n_groups : int
    qs : sequence of float
        Quantiles between 0 and 1.

    Returns
    -------
    numpy.ndarray, shape (len(qs), n_groups)
        NaN for groups without any valid value.
    """
    out = np.full((len(qs), n_groups), np.nan)
    ok = np.isfinite(values) & (weights > 0)
    if not ok.any():
        return out
    v, w, g = values[ok], weights[ok], groups[ok]
    order = np.lexsort((v, g))  # sort by group, then by value
    v, w, g = v[order], w[order], g[order]
    cum_w = np.cumsum(w)
    total = np.bincount(g, weights=w, minlength=n_groups)
    before = np.concatenate([[0.0], np.cumsum(total)[:-1]])
    first = np.searchsorted(g, np.arange(n_groups), side="left")
    last = np.searchsorted(g, np.arange(n_groups), side="right") - 1
    has = total > 0
    for k, q in enumerate(qs):
        idx = np.searchsorted(cum_w, before + q * total, side="left")
        idx = np.minimum(np.maximum(idx, first), last)
        out[k, has] = v[idx[has]]
    return out


class KoppenAccumulator:
    """Koppen-Geiger classification built up one month at a time.

    Feeding months one by one means the 24 monthly rasters never have to
    be held in memory together. Rules follow Peel et al. (2007), as used
    by Beck et al. (2018): temperate/cold boundary at 0 deg C, summer =
    the warmer of the two half-years AMJJAS / ONDJFM.

    Usage::

        acc = KoppenAccumulator(n)
        for month in range(1, 13):
            acc.update(month, tmean_month, precip_month)
        codes = acc.classify()  # index into KOPPEN_CLASSES, -1 = invalid
    """

    def __init__(self, n):
        def full(value):
            return np.full(n, value, dtype="float32")

        self.valid = np.ones(n, dtype=bool)
        self.t_sum, self.p_sum, self.n10 = full(0), full(0), full(0)
        self.t_cold, self.t_hot = full(np.inf), full(-np.inf)
        self.p_dry = full(np.inf)
        # half-year A = Apr-Sep, B = Oct-Mar; summer decided at the end
        self.t_half = {"A": full(0), "B": full(0)}
        self.p_half = {"A": full(0), "B": full(0)}
        self.p_half_dry = {"A": full(np.inf), "B": full(np.inf)}
        self.p_half_wet = {"A": full(-np.inf), "B": full(-np.inf)}
        self.months_seen = set()

    def update(self, month, tmean, precip):
        """Add one month of mean temperature (deg C) and precipitation (mm).

        Parameters
        ----------
        month : int
            1-12.
        tmean, precip : numpy.ndarray, shape (n,)
        """
        t = np.asarray(tmean, dtype="float32")
        p = np.asarray(precip, dtype="float32")
        self.valid &= np.isfinite(t) & np.isfinite(p)
        t = np.nan_to_num(t)
        p = np.nan_to_num(p)
        half = "A" if 4 <= month <= 9 else "B"
        self.t_sum += t
        self.p_sum += p
        self.n10 += (t > 10).astype("float32")
        np.minimum(self.t_cold, t, out=self.t_cold)
        np.maximum(self.t_hot, t, out=self.t_hot)
        np.minimum(self.p_dry, p, out=self.p_dry)
        self.t_half[half] += t
        self.p_half[half] += p
        np.minimum(self.p_half_dry[half], p, out=self.p_half_dry[half])
        np.maximum(self.p_half_wet[half], p, out=self.p_half_wet[half])
        self.months_seen.add(month)

    def classify(self):
        """Return the class index per element (-1 where data is missing)."""
        if self.months_seen != set(range(1, 13)):
            raise ValueError(f"Months fed: {sorted(self.months_seen)}")
        mat = self.t_sum / 12.0
        map_ = self.p_sum
        t_cold, t_hot, n10, p_dry = (self.t_cold, self.t_hot, self.n10,
                                     self.p_dry)
        a_is_summer = self.t_half["A"] >= self.t_half["B"]

        def pick(key, store):
            return np.where(a_is_summer, store["A" if key == "s" else "B"],
                            store["B" if key == "s" else "A"])

        ps, pw = pick("s", self.p_half), pick("w", self.p_half)
        ps_dry, pw_dry = pick("s", self.p_half_dry), pick("w",
                                                         self.p_half_dry)
        ps_wet, pw_wet = pick("s", self.p_half_wet), pick("w",
                                                         self.p_half_wet)

        # aridity threshold (mm): depends on when the rain falls
        p_th = np.where(pw >= 0.7 * map_, 2 * mat,
                        np.where(ps >= 0.7 * map_, 2 * mat + 28,
                                 2 * mat + 14))
        p_th = np.maximum(p_th, 0)

        # second letter for C and D: 0 = s (dry summer), 1 = w, 2 = f
        is_s = (ps_dry < 40) & (ps_dry < pw_wet / 3)
        is_w = pw_dry < ps_wet / 10
        both = is_s & is_w  # rare: keep the drier half-year
        is_s = np.where(both, ps < pw, is_s)
        is_w = np.where(both, ~is_s, is_w)
        second = np.where(is_s, 0, np.where(is_w, 1, 2))
        # third letter: 0 = a, 1 = b, 2 = c, 3 = d (d only for D)
        third = np.where(t_hot >= 22, 0,
                         np.where(n10 >= 4, 1,
                                  np.where(t_cold < -38, 3, 2)))

        codes = np.full(len(mat), -1, dtype="int16")
        polar = t_hot < 10
        arid = ~polar & (map_ < 10 * p_th)
        trop = ~polar & ~arid & (t_cold >= 18)
        temp = ~polar & ~arid & ~trop & (t_cold > 0)
        cold = ~polar & ~arid & ~trop & (t_cold <= 0)

        codes[polar] = np.where(t_hot[polar] > 0, _KG_CODE["ET"],
                                _KG_CODE["EF"])
        # BWh, BWk, BSh, BSk
        codes[arid] = (_KG_CODE["BWh"]
                       + 2 * (map_[arid] >= 5 * p_th[arid])
                       + (mat[arid] < 18))
        codes[trop] = np.where(
            p_dry[trop] >= 60, _KG_CODE["Af"],
            np.where(p_dry[trop] >= 100 - map_[trop] / 25, _KG_CODE["Am"],
                     _KG_CODE["Aw"]))
        codes[temp] = (_KG_CODE["Csa"] + 3 * second[temp]
                       + np.minimum(third[temp], 2))
        codes[cold] = _KG_CODE["Dsa"] + 4 * second[cold] + third[cold]
        codes[~self.valid] = -1
        return codes


def koppen_from_monthly(tmean, precip):
    """Classify rows of monthly climate.

    Parameters
    ----------
    tmean, precip : numpy.ndarray, shape (n, 12)
        Monthly mean temperature (deg C) and precipitation (mm), Jan-Dec.

    Returns
    -------
    numpy.ndarray of int, shape (n,)
        Index into KOPPEN_CLASSES (-1 if any month is missing).
    """
    acc = KoppenAccumulator(tmean.shape[0])
    for m in range(12):
        acc.update(m + 1, tmean[:, m], precip[:, m])
    return acc.classify()


def class_fractions(codes, weights, groups, n_groups):
    """Area fraction of every Koppen class per group.

    Returns
    -------
    numpy.ndarray, shape (n_groups, len(KOPPEN_CLASSES))
        Rows sum to 1 (NaN rows for groups without classified pixels).
    """
    k = len(KOPPEN_CLASSES)
    ok = (codes >= 0) & (weights > 0)
    counts = np.bincount(groups[ok] * k + codes[ok], weights=weights[ok],
                         minlength=n_groups * k).reshape(n_groups, k)
    total = counts.sum(axis=1, keepdims=True)
    return np.divide(counts, total, out=np.full_like(counts, np.nan),
                     where=total > 0)


# ======================================================================
# (c) Derived climate indices
# ======================================================================
DAYS_IN_MONTH = np.array([31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
MID_MONTH_DOY = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288,
                          319, 349])


def extraterrestrial_radiation_mm_day(lat_deg, doy=MID_MONTH_DOY):
    """Extraterrestrial radiation Ra (FAO-56, eq. 21) in mm/day equivalent.

    Parameters
    ----------
    lat_deg : array-like, shape (n,)
        Latitude in decimal degrees.
    doy : array-like, shape (m,)
        Day of year for which Ra is evaluated (default: mid-month).

    Returns
    -------
    numpy.ndarray, shape (n, m)
    """
    phi = np.radians(np.asarray(lat_deg, dtype=float))[:, None]
    j = np.asarray(doy, dtype=float)[None, :]
    dr = 1 + 0.033 * np.cos(2 * np.pi * j / 365)
    delta = 0.409 * np.sin(2 * np.pi * j / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    ra_mj = (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(phi) * np.sin(delta)
        + np.cos(phi) * np.cos(delta) * np.sin(ws)
    )
    return 0.408 * ra_mj  # MJ m-2 day-1 -> mm day-1


def hargreaves_annual_pet_mm(tmin, tmax, lat_deg):
    """Annual Hargreaves-Samani potential evapotranspiration (mm/yr).

    PET = 0.0023 * Ra * (Tmean + 17.8) * sqrt(Tmax - Tmin), summed over
    months. ``tmin`` and ``tmax`` have shape (n_locations, 12).
    """
    tmean = (tmin + tmax) / 2.0
    ra = extraterrestrial_radiation_mm_day(lat_deg)
    daily = (0.0023 * ra * np.clip(tmean + 17.8, 0, None)
             * np.sqrt(np.clip(tmax - tmin, 0, None)))
    return (daily * DAYS_IN_MONTH[None, :]).sum(axis=1)


def frost_days_estimate(tmin, sigma=FROST_SIGMA_C):
    """Expected number of days per year with Tmin < 0 deg C.

    Assumes daily Tmin within a month ~ Normal(monthly mean Tmin, sigma),
    so P(frost on a day) = Phi(-Tmin / sigma). Gives a smooth, continuous
    variable instead of the coarse "number of sub-zero months".
    """
    return (ndtr(-tmin / sigma) * DAYS_IN_MONTH[None, :]).sum(axis=1)

## Step 1 · SInAS — location names and diagnostic counts

**Why:** the SInAS raw `location` string is the key that every other data
stream (trade tensor, GBIF, species lists) uses. We take it from the SInAS
occurrence table itself so all streams agree.

**What:**
1. Read `SInAS_3.1.1.csv`.
2. Build `loc_names` = one `(locationID, location)` pair per location.
   If a locationID has several spellings, the first is kept and a warning
   is raised.
3. Count records / taxa per location → `diag_*` columns.

> ⚠️ `diag_*` columns are **derived from the labels the model predicts**.
> They are kept in the master table for inspection only and are excluded
> from the model-ready table.

In [6]:
print("Step 1: SInAS location names and diagnostic counts")
sinas = read_table_auto(SINAS_RAW_CSV)
sinas["locationID"] = pd.to_numeric(sinas["locationID"],
                                    errors="coerce").astype("Int64")

# --- authoritative names ------------------------------------------------
loc_names = (
    sinas[["locationID", "location"]]
    .dropna()
    .assign(location=lambda d: d["location"].astype(str).str.strip())
    .drop_duplicates()
)
multi_name = loc_names["locationID"].duplicated(keep=False)
if multi_name.any():
    n_multi = loc_names.loc[multi_name, "locationID"].nunique()
    warnings.warn(f"{n_multi} locationIDs carry >1 name; keeping first")
loc_names = loc_names.drop_duplicates("locationID")

# --- label-derived diagnostics (NEVER model inputs) ----------------------
diag = sinas.groupby("locationID").agg(
    diag_sinas_total_records=("taxon", "size"),
    diag_sinas_unique_taxa=("taxon", "nunique"),
)
if "establishmentMeans" in sinas.columns:
    means = (sinas["establishmentMeans"].fillna("").astype(str)
             .str.lower().str.strip())
    is_alien = means.str.contains(
        r"introduced|invasive|naturali[sz]ed|alien|non[\s_-]?native")
    is_native = means.str.contains(r"native|indigenous") & ~is_alien
    diag["diag_sinas_unique_alien_taxa"] = (
        sinas[is_alien].groupby("locationID")["taxon"].nunique())
    diag["diag_sinas_unique_native_taxa"] = (
        sinas[is_native].groupby("locationID")["taxon"].nunique())
diag = diag.fillna(0).reset_index()

n_sinas_locations = int(sinas["locationID"].nunique())
del sinas  # large table, free memory

print(f"   {len(loc_names)} named locations, "
      f"{n_sinas_locations} distinct locationIDs")
loc_names.head()

Step 1: SInAS location names and diagnostic counts
   SInAS_3.1.1.csv: encoding=utf-8-sig, sep=' '
   289 named locations, 289 distinct locationIDs


,locationID,location
0,101,Aegean
1409,2,Afghanistan
2947,267,Akrotiri and Dhekelia
3043,5,Åland
3130,254,Alaska


## Step 2 · Manuela's covariates — one row per location

**Why:** Manuela's file is in *long* format (one row per location ×
taxon group), so location-level variables are repeated. We need exactly one
row per location.

**What:**
1. Read the file, drop spreadsheet index columns (`Unnamed: …`).
2. Convert numeric columns, accepting decimal commas (`1,5` → `1.5`).
3. Keep static variables once per location (first value). If a "static"
   variable actually varies within a location, a warning lists it.
4. Pivot `native_richness` per taxon group into wide columns
   `native_richness_<group>`.
5. Rename Manuela's own latitude/longitude/area/seasonality to `manuela_*`.
   These are **only** used for QA in Step 5 and then dropped, because we
   recompute them consistently for every location (Steps 3–4).

In [7]:
print("Step 2: collapsing Manuela's table to one row per location")
man = read_table_auto(MANUELA_CSV)
man = man.drop(columns=[c for c in man.columns if c.startswith("Unnamed")])
man["locationID"] = pd.to_numeric(man["locationID"],
                                  errors="coerce").astype("Int64")

NUMERIC_COLS = [
    "latitude", "longitude", "temp_season", "precip_season",
    "gdp_capita_mean", "pop_dens_mean", "area_sqm", "island", "coastal",
    "years_colonial_activity", "native_richness",
]
for col in NUMERIC_COLS:
    if col in man.columns:
        man[col] = pd.to_numeric(
            man[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

STATIC_COLS = [
    c for c in [
        "ISO3", "sinas_region", "island", "coastal", "latitude",
        "longitude", "temp_season", "precip_season", "gdp_capita_mean",
        "pop_dens_mean", "area_sqm", "years_colonial_activity",
    ] if c in man.columns
]
n_varying = (man.groupby("locationID")[STATIC_COLS]
             .nunique(dropna=True).gt(1).sum())
n_varying = n_varying[n_varying > 0].to_dict()
if n_varying:
    warnings.warn("Static columns varying within a location "
                  f"(first value kept): {n_varying}")
loc_tbl = man.groupby("locationID")[STATIC_COLS].first().reset_index()

if {"taxaGroup", "native_richness"} <= set(man.columns):
    richness_wide = (
        man.pivot_table(index="locationID", columns="taxaGroup",
                        values="native_richness", aggfunc="max")
        .rename(columns=lambda g: f"native_richness_{slug(g)}")
        .reset_index()
    )
    loc_tbl = loc_tbl.merge(richness_wide, on="locationID", how="left")

# Manuela's versions of variables we recompute -> QA only (dropped later).
loc_tbl = loc_tbl.rename(columns={
    "latitude": "manuela_latitude",
    "longitude": "manuela_longitude",
    "area_sqm": "manuela_area_sqm",
    "temp_season": "manuela_temp_season",
    "precip_season": "manuela_precip_season",
})
print(f"   {len(loc_tbl)} locations x {loc_tbl.shape[1] - 1} variables")
loc_tbl.head()

Step 2: collapsing Manuela's table to one row per location
   variables_SInAS_manuela.csv: encoding=utf-8-sig, sep=','
   289 locations x 20 variables


,locationID,ISO3,sinas_region,island,coastal,manuela_latitude,manuela_longitude,manuela_temp_season,manuela_precip_season,gdp_capita_mean,pop_dens_mean,manuela_area_sqm,years_colonial_activity,native_richness_amphibians,native_richness_ants,native_richness_birds,native_richness_fishes,native_richness_insects,native_richness_mammals,native_richness_reptiles,native_richness_vascular_plants
0,1,ABW,Central America and the Caribbean,1,1,12.5,-70.0,108.779617,67.710564,23775.649744,537.331313,1.819384e+08,0,NaN,NaN,103.0,574.0,NaN,33.0,11.0,512.0
1,2,AFG,Western Asia,0,0,33.8,66.0,956.116211,93.749359,292.023734,37.083653,6.438387e+11,0,14.0,130.0,423.0,111.0,85.0,141.0,112.0,6432.0
2,3,AGO,Sub-Saharan Africa,0,1,-12.3,17.5,194.560257,94.884918,2327.077139,15.618481,1.246918e+12,400,138.0,883.0,940.0,976.0,580.0,330.0,263.0,9737.0
3,4,AIA,Central America and the Caribbean,1,1,18.2,-63.1,94.609192,36.182125,NaN,NaN,7.988243e+07,0,NaN,NaN,105.0,429.0,NaN,NaN,15.0,NaN
4,5,ALA,Europe,1,1,60.2,20.1,697.126526,31.913948,NaN,NaN,1.506012e+09,0,NaN,NaN,180.0,NaN,NaN,NaN,NaN,NaN


## Step 3 · Polygons — geometry metrics and merge

**Why:** polygons let us compute geography (area, perimeter, centroid) and
climate (Step 4) the *same way* for every location.

**What:**
1. Read `SInAS_locations_3.1.gpkg`; dissolve if a locationID has several
   polygons; reproject to WGS84 (EPSG:4326) if needed.
2. In the equal-area projection EPSG:6933 compute:
   `polygon_area_sqkm`, `polygon_perimeter_km`, and the centroid →
   `latitude`, `longitude`, `abs_latitude`.
3. Left-join names (Step 1), Manuela covariates (Step 2) and diagnostics.

In [8]:
print("Step 3: polygon geometry and merge")
gdf = gpd.read_file(GPKG_FILE)
if "locationID" not in gdf.columns:
    raise KeyError(f"'locationID' not in GPKG columns: {list(gdf.columns)}")
gdf["locationID"] = pd.to_numeric(gdf["locationID"],
                                  errors="coerce").astype("Int64")
gdf = gdf.rename(columns={"location": "location_gpkg"})
gdf = gdf[[c for c in ("locationID", "location_gpkg", "geometry")
           if c in gdf.columns]]
if not gdf["locationID"].is_unique:
    gdf = gdf.dissolve(by="locationID", as_index=False)
if gdf.crs is None:
    raise ValueError("GPKG has no CRS")
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs("EPSG:4326")

equal_area = gdf.geometry.to_crs("EPSG:6933")
gdf["polygon_area_sqkm"] = equal_area.area / 1e6
gdf["polygon_perimeter_km"] = equal_area.length / 1e3
centroid = equal_area.centroid.to_crs("EPSG:4326")
gdf["latitude"] = centroid.y
gdf["longitude"] = centroid.x
gdf["abs_latitude"] = gdf["latitude"].abs()

gdf = (gdf.merge(loc_names, on="locationID", how="left")
          .merge(loc_tbl, on="locationID", how="left")
          .merge(diag, on="locationID", how="left")
          .reset_index(drop=True))
print(f"   {len(gdf)} polygons | {gdf['location'].notna().sum()} with a "
      f"SInAS name | {n_sinas_locations} SInAS locations")

Step 3: polygon geometry and merge
   289 polygons | 289 with a SInAS name | 289 SInAS locations


## Step 4 · Climate

Step 4 is split into four cells:

| Cell | What it does |
|---|---|
| 4a | WorldClim zonal statistics: means for all 58 layers, area-weighted quantiles for the key variables |
| 4b | CHELSA gap-fill for any location WorldClim does not cover (e.g. Saint Helena) |
| 4c | Turn the results into climate columns + derived indices (frost days, PET, aridity) |
| 4d | Within-country climate **distribution**: quantile columns and Köppen–Geiger class fractions |

**Why distributions and not only means?** A large or varied country is badly
described by one average: Chile's mean climate matches almost none of Chile.
For invasion risk what matters is whether *some* part of a country suits a
species. Quantiles (e.g. the 5 % coldest winters) and the area share of each
Köppen–Geiger climate class keep that information, in a small fixed-length
vector per country.

**Weighting.** Every pixel counts by the fraction of it inside the polygon ×
cos(latitude), i.e. by the land area it represents.

### 4a · WorldClim zonal statistics

1. Locate the rasters (`bio` 1–19, monthly `tmin`, `tmax`, `prec`, `elev`);
   download / unzip them first if needed.
2. **Simplify the polygons** to `SIMPLIFY_DEG` (~500 m). Coastline detail
   finer than a pixel only costs time. Polygons that would collapse (tiny
   islands) keep their original shape.
3. **Coverage index (one pass):** record which pixels each polygon covers
   and what fraction of each pixel lies inside it. This is the slow,
   geometric part, so it is done once and reused for every layer.
4. **Every layer (seconds each):** area-weighted mean per polygon, plus
   p05 / p50 / p95 for the `QUANTILE_VARS`. Polygons with no valid land
   pixel (atolls) get the **nearest valid pixel**, recorded in
   `qa_wc_extract_method` and `qa_wc_fallback_distance_km`.

⏱️ Each sub-step prints the elapsed time (about 2 minutes in total). The
first run may spend a few minutes unzipping rasters; afterwards set
`DOWNLOAD_RASTERS = False`.

In [9]:
import time

print("Step 4a: WorldClim 2.1 zonal statistics")
t0 = time.perf_counter()


def _tick(msg):
    """Print a step label with the time elapsed since Step 4a started."""
    print(f"   [{time.perf_counter() - t0:6.1f} s] {msg}", flush=True)


# --- rasters -----------------------------------------------------------------
_tick("checking rasters ...")
ensure_rasters()
rasters = index_rasters()

# --- points inside polygons + simplified polygons ----------------------------
_tick("representative points ...")
# representative_point() is guaranteed to lie inside the polygon
# (a centroid can fall in the sea for crescent-shaped coastlines).
rep_pts = gdf.geometry.representative_point()

_tick(f"simplifying polygons (tolerance {SIMPLIFY_DEG} deg) ...")
if SIMPLIFY_DEG > 0:
    simple = gdf.geometry.simplify(SIMPLIFY_DEG, preserve_topology=False)
    broken = simple.is_empty | ~simple.is_valid
    simple[broken] = gdf.geometry[broken]
else:
    simple, broken = gdf.geometry, pd.Series(False, index=gdf.index)
zones = gpd.GeoDataFrame(geometry=simple, crs=gdf.crs)
try:
    _tick(f"vertices {gdf.geometry.count_coordinates().sum():,} -> "
          f"{zones.geometry.count_coordinates().sum():,} "
          f"({int(broken.sum())} polygons kept unsimplified)")
except AttributeError:  # geopandas < 0.13
    pass

# --- pixel coverage per polygon: ONCE ----------------------------------------
_tick("computing pixel coverage per polygon (one pass) ...")
cov_index = build_coverage_index(zones, rasters["bio"][1])
_tick(f"coverage index: {len(cov_index['cell_id']):,} polygon-pixel pairs")

# --- every layer: fast numpy aggregation -------------------------------------
# 19 bio + 3 elev + 12 tmin + 12 tmax + 12 prec = 58 layers.
QUANT_LAYERS = {("bio", i) for i in QUANTILE_VARS}
jobs = (
    [("bio", i, "mean") for i in range(1, 20)]
    + [("elev", None, stat) for stat in ("mean", "min", "max")]
    + [(var, m, "mean") for var in ("tmin", "tmax", "prec")
       for m in range(1, 13)]
)
results = {}  # (layer, idx, stat) -> (values, fallback_used, distance)
quant = {}  # (layer, idx) -> array (len(QUANTILES), n_polygons)
pbar = tqdm(jobs, desc="Zonal statistics", unit="layer")
for layer, idx, stat in pbar:
    label = f"{layer}{idx:02d}" if idx else f"{layer}_{stat}"
    pbar.set_postfix_str(label)
    path = rasters[layer][idx] if idx else rasters[layer]
    qs = QUANTILES if (layer, idx) in QUANT_LAYERS else ()
    vals, used_fb, dist, qvals = zonal_from_index(path, cov_index, stat,
                                                  rep_pts, qs)
    results[(layer, idx, stat)] = (vals, used_fb, dist)
    if qvals is not None:
        quant[(layer, idx)] = qvals

wc_missing = np.isnan(results[("bio", 1, "mean")][0])
_tick(f"done; {int(wc_missing.sum())} location(s) without WorldClim data: "
      f"{gdf.loc[wc_missing, 'location'].tolist()}")

Step 4a: WorldClim 2.1 zonal statistics
   [   0.0 s] checking rasters ...
   [   0.0 s] representative points ...
   [   1.3 s] simplifying polygons (tolerance 0.005 deg) ...
   [   9.8 s] vertices 35,888,104 -> 754,193 (1 polygons kept unsimplified)
   [   9.8 s] computing pixel coverage per polygon (one pass) ...


.: 100%|██████████| 100.0/100 [00:08<00:00, 11.50it/s]              


   [  19.2 s] coverage index: 12,599,669 polygon-pixel pairs


Zonal statistics:   0%|          | 0/58 [00:00<?, ?layer/s]

   [ 121.4 s] done; 1 location(s) without WorldClim data: ['Saint Helena']


### 4b · CHELSA gap-fill

Polygons that still have no WorldClim value (outside its land mask, e.g.
Saint Helena) get every bio variable and the monthly Tmin / Tmax /
precipitation from **CHELSA V2.1 (1981–2010)**, read directly from the CHELSA
server (only the pixels of those polygons, nothing to download by hand).
For these locations the CHELSA *pixels* are also kept, so their quantiles
and Köppen–Geiger fractions (Step 4d) are computed in the same way as for
WorldClim.

As a check, CHELSA is also read for the `CHELSA_N_CONTROLS` nearest islands
that WorldClim *does* cover, and the two sources are compared. Wrong units
or a large bias between the datasets show up here. Elevation is **not**
gap-filled (CHELSA has no elevation layer).

In [10]:
print("Step 4b: CHELSA gap-fill")
gapfilled = np.zeros(len(gdf), dtype=bool)
gap_pixels = {}  # (layer, idx) -> list of (values, weights) per target
targets = np.where(wc_missing)[0]
chelsa_report = {"locations": [], "controls": [], "comparison": {}}

if CHELSA_GAPFILL and len(targets):
    # nearest WorldClim-covered islands, for the QA comparison
    has_wc = ~wc_missing & (gdf["island"] == 1).to_numpy()
    controls = []
    for t in targets:
        lat0 = gdf["latitude"].iloc[t]
        d = np.hypot(gdf["latitude"].to_numpy() - lat0,
                     (gdf["longitude"].to_numpy()
                      - gdf["longitude"].iloc[t])
                     * np.cos(np.radians(lat0)))
        d[~has_wc] = np.inf
        controls += list(np.argsort(d)[:CHELSA_N_CONTROLS])
    controls = sorted(set(controls))
    chelsa_report["locations"] = gdf["location"].iloc[targets].tolist()
    chelsa_report["controls"] = gdf["location"].iloc[controls].tolist()
    print(f"   targets: {chelsa_report['locations']}\n"
          f"   controls: {chelsa_report['controls']}")

    target_geoms = list(gdf.geometry.iloc[targets])
    control_geoms = list(gdf.geometry.iloc[controls])
    fill_jobs = ([("bio", i) for i in range(1, 20)]
                 + [(v, m) for v in ("tmin", "tmax", "prec")
                    for m in range(1, 13)])
    for layer, idx in tqdm(fill_jobs, desc="CHELSA gap-fill", unit="layer"):
        is_bio = layer == "bio"
        if layer in ("tmin", "tmax") or (layer, idx) in TEMPERATURE_LAYERS:
            kind = "temperature"
        elif layer == "prec":
            kind = "precip_month"
        else:
            kind = None
        geoms = target_geoms + (control_geoms if is_bio else [])
        pix = chelsa_polygon_pixels(chelsa_url(layer, idx), geoms, kind)
        means = np.array([weighted_mean(v, w) for v, w in pix])
        nt = len(targets)
        results[(layer, idx, "mean")][0][targets] = means[:nt]
        if not is_bio or (layer, idx) in QUANT_LAYERS:
            gap_pixels[(layer, idx)] = pix[:nt]
        if (layer, idx) in QUANT_LAYERS:
            for j, t in enumerate(targets):
                v, w = pix[j]
                quant[(layer, idx)][:, t] = weighted_group_quantiles(
                    v, w, np.zeros(len(v), dtype=int), 1, QUANTILES)[:, 0]
        if is_bio:  # compare CHELSA vs WorldClim on the control islands
            wc_ctrl = results[("bio", idx, "mean")][0][controls]
            ch_ctrl = means[nt:]
            chelsa_report["comparison"][BIOCLIM_NAMES[idx]] = {
                "worldclim_mean": float(np.mean(wc_ctrl)),
                "chelsa_mean": float(np.mean(ch_ctrl)),
                "mean_diff_chelsa_minus_wc": float(
                    np.mean(ch_ctrl - wc_ctrl)),
            }
    gapfilled = wc_missing & np.isfinite(results[("bio", 1, "mean")][0])
    print("   CHELSA vs WorldClim on the control islands (raw units, "
          "BIO4 still x100):")
    print(pd.DataFrame(chelsa_report["comparison"]).T.round(2).to_string())
    print(f"   gap-filled {int(gapfilled.sum())} location(s)")
else:
    print("   nothing to gap-fill")

Step 4b: CHELSA gap-fill
   targets: ['Saint Helena']
   controls: ['Ascension', 'Tristan da Cunha', 'São Tomé and Príncipe']


CHELSA gap-fill:   0%|          | 0/55 [00:00<?, ?layer/s]

   CHELSA vs WorldClim on the control islands (raw units, BIO4 still x100):
                                      worldclim_mean  chelsa_mean  mean_diff_chelsa_minus_wc
wc_bio01_annual_mean_temp_C                    18.18        19.99                       1.81
wc_bio02_mean_diurnal_range_C                   5.97         1.41                      -4.56
wc_bio03_isothermality                         60.01         0.26                     -59.75
wc_bio04_temp_seasonality_sd_C                146.79       144.99                      -1.80
wc_bio05_max_temp_warmest_month_C              23.42        22.91                      -0.51
wc_bio06_min_temp_coldest_month_C              13.34        17.36                       4.02
wc_bio07_temp_annual_range_C                   10.08         5.55                      -4.53
wc_bio08_mean_temp_wettest_quarter_C           17.82        20.21                       2.39
wc_bio09_mean_temp_driest_quarter_C            18.84        20.09                      

### 4c · Climate columns and derived indices

1. The 19 BIOCLIM means become `wc_bio01` … `wc_bio19` columns. BIO4 is
   stored by WorldClim (and CHELSA) as SD × 100; we divide by 100 → °C.
2. `qa_wc_extract_method` records how each row was obtained:
   `polygon_mean`, `nearest_valid_pixel`, `chelsa_gapfill` or `missing`.
3. Elevation: mean, min, max and relief (max − min).
4. Derived variables from monthly Tmin/Tmax:
   * `wc_subzero_tmin_months` — months with mean Tmin < 0 °C;
   * `wc_annual_frost_days_est` — expected frost days (see helper);
   * `wc_pet_hargreaves_mm` — Hargreaves-Samani PET using the centroid
     latitude;
   * `wc_aridity_index_p_pet` — BIO12 / max(PET, `PET_FLOOR_MM`).

In [11]:
print("Step 4c: climate columns and derived indices")
for i in range(1, 20):
    vals = results[("bio", i, "mean")][0]
    if i == 4:
        vals = vals / 100.0  # BIO4 is stored as SD x 100
    gdf[BIOCLIM_NAMES[i]] = vals

# All layers share one land mask -> record the fallback from BIO1 only.
_, fallback_used, fallback_dist = results[("bio", 1, "mean")]
gdf["qa_wc_extract_method"] = np.select(
    [gapfilled, ~np.isfinite(gdf["wc_bio01_annual_mean_temp_C"]),
     fallback_used],
    ["chelsa_gapfill", "missing", "nearest_valid_pixel"],
    default="polygon_mean",
)
gdf["qa_wc_fallback_distance_km"] = fallback_dist * PX_KM

# --- elevation ---------------------------------------------------------------
for stat in ("mean", "min", "max"):
    gdf[f"wc_elevation_{stat}_m"] = results[("elev", None, stat)][0]
gdf["wc_elevation_relief_m"] = (gdf["wc_elevation_max_m"]
                                - gdf["wc_elevation_min_m"])

# --- monthly Tmin / Tmax and derived indices ---------------------------------
tmin = np.column_stack([results[("tmin", m, "mean")][0]
                        for m in range(1, 13)])
tmax = np.column_stack([results[("tmax", m, "mean")][0]
                        for m in range(1, 13)])
prec = np.column_stack([results[("prec", m, "mean")][0]
                        for m in range(1, 13)])
incomplete = np.isnan(tmin).any(axis=1) | np.isnan(tmax).any(axis=1)

gdf["wc_subzero_tmin_months"] = np.where(incomplete, np.nan,
                                         (tmin < 0).sum(axis=1))
gdf["wc_annual_frost_days_est"] = np.where(incomplete, np.nan,
                                           frost_days_estimate(tmin))
pet = np.where(
    incomplete, np.nan,
    hargreaves_annual_pet_mm(tmin, tmax, gdf["latitude"].to_numpy()),
)
gdf["wc_pet_hargreaves_mm"] = pet
gdf["wc_aridity_index_p_pet"] = (gdf["wc_bio12_annual_precipitation_mm"]
                                 / np.clip(pet, PET_FLOOR_MM, None))
print(gdf["qa_wc_extract_method"].value_counts().to_string())

Step 4c: climate columns and derived indices
qa_wc_extract_method
polygon_mean      288
chelsa_gapfill      1


### 4d · Within-country climate distribution

**Quantiles.** For each variable in `QUANTILE_VARS` the area-weighted
p05 / p50 / p95 across the country's pixels, e.g.
`wc_bio06_min_temp_coldest_month_C_p05` = the winter minimum below which the
coldest 5 % of the country lies. Useful for physiological limits (frost,
heat, drought) that a mean hides.

**Köppen–Geiger fractions.** Every pixel is classified into one of the 30
Köppen–Geiger classes from its monthly mean temperature ((Tmin + Tmax) / 2)
and precipitation, following the rules of Peel et al. (2007) as used by
Beck et al. (2018) (temperate / cold boundary at 0 °C; summer = the warmer
half-year). `kg_<class>` = area fraction of the country in that class
(rows sum to 1); `kg_dominant_class` = the class with the largest share.
The 12 monthly layers are fed in one month at a time, so they never need to
be in memory together.

How each row's fractions were obtained is stored in `qa_kg_method`:
`worldclim_pixels`, `chelsa_pixels` (gap-filled locations),
`polygon_mean_climate` (atolls: one class from the polygon's mean monthly
climate) or `missing`.

> These fractions are also exported on their own as
> `country_koppen_fractions.csv`. In the model notebook a species' climate
> profile can be built from the fractions of the countries where it occurs
> (GBIF presence matrix, leaving the scored country out) and compared with
> the target country's fractions.

In [12]:
print("Step 4d: within-country climate distribution")
n_poly = len(gdf)

# --- quantile columns --------------------------------------------------------
QUANT_COLS = []
for i in QUANTILE_VARS:
    qv = quant[("bio", i)] / (100.0 if i == 4 else 1.0)
    for k, q in enumerate(QUANTILES):
        col = f"{BIOCLIM_NAMES[i]}_p{round(q * 100):02d}"
        gdf[col] = qv[k]
        QUANT_COLS.append(col)
print(f"   {len(QUANT_COLS)} quantile columns")

# --- Koppen-Geiger from WorldClim pixels -------------------------------------
cells, inverse = np.unique(cov_index["cell_id"], return_inverse=True)
acc = KoppenAccumulator(len(cells))
for m in tqdm(range(1, 13), desc="Koppen-Geiger (WorldClim)", unit="month"):
    t_month = (read_grid(rasters["tmin"][m], cov_index)[cells]
               + read_grid(rasters["tmax"][m], cov_index)[cells]) / 2.0
    p_month = read_grid(rasters["prec"][m], cov_index)[cells]
    acc.update(m, t_month, p_month)
cell_codes = acc.classify()
del acc, t_month, p_month
kg = class_fractions(cell_codes[inverse].astype(np.int64),
                     cov_index["area_w"], cov_index["owner"], n_poly)
kg_method = np.where(np.isfinite(kg[:, 0]), "worldclim_pixels", "missing")

# --- gap-filled locations: CHELSA pixels -------------------------------------
for j, t in enumerate(targets if gapfilled.any() else []):
    w = gap_pixels[("tmin", 1)][j][1]
    sub = KoppenAccumulator(len(w))
    for m in range(1, 13):
        t_month = (gap_pixels[("tmin", m)][j][0]
                   + gap_pixels[("tmax", m)][j][0]) / 2.0
        sub.update(m, t_month, gap_pixels[("prec", m)][j][0])
    kg[t] = class_fractions(sub.classify().astype(np.int64), w,
                            np.zeros(len(w), dtype=np.int64), 1)[0]
    kg_method[t] = "chelsa_pixels"

# --- remaining rows with a mean climate (atoll fallback): one class ----------
mean_only = ((kg_method == "missing") & ~np.isnan(tmin).any(axis=1)
             & ~np.isnan(tmax).any(axis=1) & ~np.isnan(prec).any(axis=1))
if mean_only.any():
    rows = np.where(mean_only)[0]
    codes = koppen_from_monthly((tmin[rows] + tmax[rows]) / 2.0, prec[rows])
    for r, c in zip(rows, codes):
        if c >= 0:
            kg[r] = 0.0
            kg[r, c] = 1.0
            kg_method[r] = "polygon_mean_climate"

KG_COLS = [f"kg_{c}" for c in KOPPEN_CLASSES]
gdf[KG_COLS] = kg
gdf["kg_dominant_class"] = pd.Series(
    np.where(np.isfinite(kg[:, 0]),
             np.array(KOPPEN_CLASSES, dtype=object)[np.nan_to_num(kg)
                                                   .argmax(axis=1)],
             None),
    index=gdf.index)
gdf["qa_kg_method"] = kg_method
del cov_index, cells, inverse, cell_codes, results, quant, gap_pixels

print(f"   Koppen-Geiger: {gdf['qa_kg_method'].value_counts().to_dict()}")
print("   most common dominant classes:",
      gdf["kg_dominant_class"].value_counts().head(8).to_dict())
_tick("Step 4 done")

Step 4d: within-country climate distribution
   15 quantile columns


Koppen-Geiger (WorldClim):   0%|          | 0/12 [00:00<?, ?month/s]

   Koppen-Geiger: {'worldclim_pixels': 288, 'chelsa_pixels': 1}
   most common dominant classes: {'Aw': 54, 'Af': 49, 'BWh': 36, 'Cfb': 25, 'Dfb': 24, 'Csa': 17, 'Am': 16, 'BSh': 12}
   [ 243.8 s] Step 4 done


## Step 5 · Quality assurance

Checks written to `country_features_qa.json`:

| Check | What to look for |
|---|---|
| `sinas_locations_without_polygon` | SInAS locations that will have **no** feature row |
| `gpkg_name_differs_from_sinas_name` | Polygon name ≠ SInAS name (we use the SInAS one) |
| `new_vs_manuela_worldclim` | Pearson r between our BIO4/BIO15 and Manuela's — should be > 0.95 (she used 30″ rasters, we use 2.5′, so small differences are expected) |
| `model_locations_missing_from_master` | Locations used by the model without a feature row (they would get a zero vector) |
| `manuela_area_over_polygon_area_median_ratio` | ≈ 1 if Manuela's `area_sqm` is really m² |
| `chelsa_gapfill` | Locations filled from CHELSA, their control islands and the CHELSA − WorldClim difference per BIO variable on those controls |
| `wc_missing_locations` | Locations without climate even after the fallback and gap-fill |
| `koppen_method_counts` | How the Köppen–Geiger fractions were obtained (WorldClim pixels, CHELSA pixels, polygon-mean climate, missing) |
| `koppen_dominant_class_counts` | Number of locations per dominant Köppen–Geiger class (sanity check) |
| `nan_counts` | Remaining missing values per column |

After the checks, all `manuela_*` and helper columns are dropped so the
master holds each variable exactly once.

In [13]:
print("Step 5: quality assurance")
qa = {
    "n_polygons": int(len(gdf)),
    "n_sinas_locations": n_sinas_locations,
    "sinas_locations_without_polygon":
        int(n_sinas_locations - gdf["location"].notna().sum()),
    "worldclim_resolution": WC_RES,
}

# --- polygon name vs SInAS name ----------------------------------------------
if "location_gpkg" in gdf.columns:
    differs = (gdf["location_gpkg"].notna() & gdf["location"].notna()
               & (gdf["location_gpkg"].astype(str).str.strip()
                  != gdf["location"]))
    qa["gpkg_name_differs_from_sinas_name"] = (
        gdf.loc[differs, ["locationID", "location_gpkg", "location"]]
        .astype(str).values.tolist())

# --- cross-check against Manuela's WorldClim BIO4 / BIO15 -------------------
manuela_cmp = {}
for ours, theirs, scale in (
    ("wc_bio04_temp_seasonality_sd_C", "manuela_temp_season", 100.0),
    ("wc_bio15_precip_seasonality_cv", "manuela_precip_season", 1.0),
):
    if theirs not in gdf.columns:
        continue
    pair = pd.DataFrame({"ours": gdf[ours] * scale,
                         "theirs": gdf[theirs]}).dropna()
    if len(pair) > 2:
        ratio = pair["ours"] / pair["theirs"].replace(0, np.nan)
        manuela_cmp[ours] = {
            "n": int(len(pair)),
            "pearson_r": float(pair["ours"].corr(pair["theirs"])),
            "MAE": float((pair["ours"] - pair["theirs"]).abs().mean()),
            "median_ratio_ours_over_theirs": float(ratio.median()),
        }
qa["new_vs_manuela_worldclim"] = manuela_cmp
if manuela_cmp:
    print(pd.DataFrame(manuela_cmp).T.round(3).to_string())

# --- coverage of the names used by the model ---------------------------------
model_species_csv = first_existing(*MODEL_SPECIES_CANDIDATES)
if model_species_csv is not None:
    model_names = (pd.read_csv(model_species_csv, usecols=["location"])
                   ["location"].dropna().astype(str).unique())
    keys = gdf["location"].map(lambda s: norm_name(s) if pd.notna(s)
                               else None)
    missing = sorted(set(model_names) - set(gdf["location"].dropna()))
    near = {m: gdf.loc[keys == norm_name(m), "location"].tolist()
            for m in missing if (keys == norm_name(m)).any()}
    qa["model_locations"] = int(len(model_names))
    qa["model_locations_missing_from_master"] = missing
    qa["near_matches_whitespace_or_case"] = near
    print(f"   model locations: {len(model_names)} | "
          f"missing from master: {len(missing)}")
    if missing:
        warnings.warn(f"{len(missing)} model locations have NO feature "
                      f"row: {missing[:15]}")
else:
    print("   (no sinas_matched_species.csv found - coverage check skipped)")

# --- area unit check ---------------------------------------------------------
if "manuela_area_sqm" in gdf.columns:
    ratio = (gdf["manuela_area_sqm"] / (gdf["polygon_area_sqkm"] * 1e6))
    ratio = ratio.replace([np.inf, -np.inf], np.nan).dropna()
    ratio = ratio[ratio > 0]
    if len(ratio):
        qa["manuela_area_over_polygon_area_median_ratio"] = float(
            ratio.median())
        print(f"   median Manuela area / polygon area = "
              f"{ratio.median():.3g} (expect ~1)")

# --- CHELSA gap-fill report --------------------------------------------------
qa["chelsa_gapfill"] = chelsa_report

# --- Koppen-Geiger -----------------------------------------------------------
qa["koppen_method_counts"] = gdf["qa_kg_method"].value_counts().to_dict()
qa["koppen_dominant_class_counts"] = (
    gdf["kg_dominant_class"].value_counts().to_dict())

# --- climate coverage --------------------------------------------------------
qa["wc_extract_method_counts"] = (
    gdf["qa_wc_extract_method"].value_counts().to_dict())
qa["wc_missing_locations"] = gdf.loc[
    gdf["qa_wc_extract_method"] == "missing", "location"].tolist()
if qa["wc_missing_locations"]:
    warnings.warn("No climate value even after fallback: "
                  f"{qa['wc_missing_locations']}")

# --- drop QA-only / duplicate columns ----------------------------------------
gdf = gdf.drop(columns=["location_gpkg"]
               + [c for c in gdf.columns if c.startswith("manuela_")],
               errors="ignore")
na_counts = gdf.drop(columns="geometry").isna().sum()
qa["nan_counts"] = {k: int(v) for k, v in na_counts.items() if v}
pd.Series(qa["nan_counts"], name="n_missing").to_frame()

Step 5: quality assurance
                                    n  pearson_r    MAE  median_ratio_ours_over_theirs
wc_bio04_temp_seasonality_sd_C  285.0        1.0  1.591                          0.999
wc_bio15_precip_seasonality_cv  285.0        1.0  0.310                          0.999
   model locations: 289 | missing from master: 0
   median Manuela area / polygon area = 1 (expect ~1)


,n_missing
gdp_capita_mean,51
pop_dens_mean,53
native_richness_amphibians,160
native_richness_ants,101
native_richness_birds,6
native_richness_fishes,39
native_richness_insects,101
native_richness_mammals,100
native_richness_reptiles,90
native_richness_vascular_plants,92


## Step 6 · Write outputs

### 6a · Master table
All variables, ordered as: identifiers → geography → socio-economy &
biodiversity → climate means → climate distribution (quantiles, Köppen–Geiger
fractions, dominant class) → extra elevation → `qa_*` → `diag_*`.
Saved both as CSV and as GeoPackage (with geometry, for mapping).

In [14]:
print("Step 6a: master table")
richness_cols = sorted(c for c in gdf.columns
                       if c.startswith("native_richness_"))

ID_COLS = ["locationID", "location", "ISO3", "sinas_region"]
GEO = ["latitude", "longitude", "abs_latitude", "polygon_area_sqkm",
       "polygon_perimeter_km", "island", "coastal",
       "wc_elevation_mean_m", "wc_elevation_relief_m"]
CLIM = list(BIOCLIM_NAMES.values()) + [
    "wc_subzero_tmin_months", "wc_annual_frost_days_est",
    "wc_pet_hargreaves_mm", "wc_aridity_index_p_pet"]
SOCIO = (["gdp_capita_mean", "pop_dens_mean", "years_colonial_activity"]
         + richness_cols)
DIST = QUANT_COLS + KG_COLS  # within-country climate distribution
ELEV_EXTRA = ["wc_elevation_min_m", "wc_elevation_max_m"]
QA_COLS = ["qa_wc_extract_method", "qa_wc_fallback_distance_km",
           "qa_kg_method"]
DIAG_COLS = [c for c in gdf.columns if c.startswith("diag_")]

ordered = list(dict.fromkeys(
    c for c in ID_COLS + GEO + SOCIO + CLIM + DIST
    + ["kg_dominant_class"] + ELEV_EXTRA + QA_COLS + DIAG_COLS
    if c in gdf.columns))
rest = [c for c in gdf.columns if c not in ordered and c != "geometry"]
final_order = ordered + rest

master = gdf.sort_values("location").reset_index(drop=True)
pd.DataFrame(master[final_order]).to_csv(CSV_OUT, index=False)
master[final_order + ["geometry"]].to_file(GPKG_OUT, driver="GPKG")
print(f"   {CSV_OUT.name}: {len(master)} rows x {len(final_order)} columns")
if rest:
    print(f"   unclassified columns appended at the end: {rest}")

# Koppen-Geiger fractions on their own (raw, unimputed) for the model
# notebook, where species climate profiles are built from them.
master[["locationID", "location"] + KG_COLS
       + ["kg_dominant_class", "qa_kg_method"]].to_csv(KOPPEN_OUT,
                                                       index=False)
print(f"   {KOPPEN_OUT.name}: {len(master)} rows x {len(KG_COLS)} classes")

Step 6a: master table
   complete_country_features_master.csv: 289 rows x 102 columns
   country_koppen_fractions.csv: 289 rows x 30 classes


### 6b · Model-ready table

Transformations applied (in this order):
1. Keep only named locations and feature columns (no `diag_*`, no `qa_*`).
2. `log1p` on right-skewed, non-negative variables (`LOG1P` list, incl. the
   precipitation quantiles); columns are renamed `log1p_<name>`. Köppen–
   Geiger fractions stay as they are (0–1).
3. Drop constant columns (no information), e.g. Köppen classes that occur
   in no location.
4. Median-impute remaining NaNs and add a 0/1 `<col>_missing` indicator.

Scaling is deliberately left to the model pipeline.

In [15]:
print("Step 6b: model-ready table")
LOG1P = (["polygon_area_sqkm", "polygon_perimeter_km",
          "wc_elevation_relief_m", "wc_pet_hargreaves_mm",
          "wc_aridity_index_p_pet", "gdp_capita_mean", "pop_dens_mean"]
         + [BIOCLIM_NAMES[i] for i in (12, 13, 14, 16, 17, 18, 19)]
         + [c for c in QUANT_COLS
            if c.startswith((BIOCLIM_NAMES[12], BIOCLIM_NAMES[14]))]
         + richness_cols)

named = master[master["location"].notna()].copy()
feat_cols = [c for c in GEO + CLIM + DIST + SOCIO if c in named.columns]
feat = named[["locationID", "location"] + feat_cols].copy()

log_cols = [c for c in LOG1P if c in feat.columns]
feat[log_cols] = np.log1p(feat[log_cols].clip(lower=0))
feat = feat.rename(columns={c: f"log1p_{c}" for c in log_cols})

value_cols = list(feat.columns[2:])
constant = [c for c in value_cols if feat[c].nunique(dropna=True) <= 1]
feat = feat.drop(columns=constant)

imputed = {}
for col in list(feat.columns[2:]):
    n_na = int(feat[col].isna().sum())
    if n_na:
        imputed[col] = n_na
        feat[f"{col}_missing"] = feat[col].isna().astype(float)
        feat[col] = feat[col].fillna(feat[col].median())

feat.to_csv(MODEL_FEATURES_OUT, index=False)
print(f"   {MODEL_FEATURES_OUT.name}: {feat.shape[0]} locations x "
      f"{feat.shape[1] - 2} features | imputed columns: {len(imputed)}")
feat.describe().T

Step 6b: model-ready table
   country_features_model.csv: 289 locations x 100 features | imputed columns: 12


,count,mean,std,min,25%,50%,75%,max
locationID,289.0,299.16609,761.388936,1.0,73.0,145.0,217.0,3925.0
latitude,289.0,15.581417,27.686724,-75.48236,-0.831351,16.571549,38.341569,78.703395
longitude,289.0,12.801857,75.170489,-177.98826,-41.669865,17.544588,51.191218,178.773928
abs_latitude,289.0,26.604879,17.317649,0.436356,12.821179,22.141367,40.275323,78.703395
log1p_polygon_area_sqkm,289.0,9.689498,3.629226,0.423124,6.548254,10.42756,12.652043,16.644351
...,...,...,...,...,...,...,...,...
log1p_native_richness_fishes_missing,289.0,0.134948,0.342261,0.0,0.0,0.0,0.0,1.0
log1p_native_richness_insects_missing,289.0,0.349481,0.477633,0.0,0.0,0.0,1.0,1.0
log1p_native_richness_mammals_missing,289.0,0.346021,0.476525,0.0,0.0,0.0,1.0,1.0
log1p_native_richness_reptiles_missing,289.0,0.311419,0.463877,0.0,0.0,0.0,1.0,1.0


### 6c · Schema and QA report

The schema groups features (geography / climate means / climate
distribution / socio-economy / missing flags) so the model can build its country branch, and logs every
transformation. **Set `n_co_features` in the model to the total number of
feature columns listed here.**

In [16]:
print("Step 6c: schema and QA report")


def _group(cols):
    """Columns of `feat` that come from `cols` (raw or log1p name)."""
    names = set(cols) | {f"log1p_{c}" for c in cols}
    return [c for c in feat.columns if c in names]


schema = {
    "key": "location",
    "n_locations": int(len(feat)),
    "n_features": int(feat.shape[1] - 2),
    "feature_groups": {
        "geography": _group(GEO),
        "climate": _group(CLIM),
        "climate_distribution": _group(DIST),
        "socioeconomic_and_biodiversity": _group(SOCIO),
        "missing_flags": [c for c in feat.columns
                          if c.endswith("_missing")],
    },
    "log1p_transformed_source_columns": log_cols,
    "dropped_constant_columns": constant,
    "median_imputed_counts": imputed,
    "excluded_as_label_derived": DIAG_COLS,
    "koppen_classes": KOPPEN_CLASSES,
    "quantile_variables": [BIOCLIM_NAMES[i] for i in QUANTILE_VARS],
    "quantiles": list(QUANTILES),
    "notes": [
        "Scale inside the model pipeline (fit on training locations "
        "only to avoid leakage).",
        "n_co_features in the model = n_features.",
        "wc_annual_frost_days_est = expected days/yr with Tmin < 0 under "
        "N(monthly mean Tmin, FROST_SIGMA_C); aridity = P / Hargreaves PET.",
        "Pixels are area-weighted (covered fraction x cos(latitude)). "
        "Koppen-Geiger classes follow Peel et al. (2007) / Beck et al. "
        "(2018), computed from WorldClim monthly mean T and P.",
        "Climate = WorldClim 2.1, 1970-2000 means, 2.5 arc-min. "
        "Socio-economic means 2000-2020; colonial years cumulative.",
    ],
}
with open(SCHEMA_OUT, "w") as fh:
    json.dump(schema, fh, indent=2)
with open(QA_OUT, "w") as fh:
    json.dump(qa, fh, indent=2,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(f"   wrote {SCHEMA_OUT.name} and {QA_OUT.name}\nDONE")

Step 6c: schema and QA report
   wrote country_features_schema.json and country_features_qa.json
DONE
